In [1]:
from me_r_sy.ingestion.grib import get_grib , get_var
from me_r_sy.visualization.plots import Plotter
from me_r_sy.analysis.statistics import Statistics
import geopandas as gpd
import numpy as np
import yaml
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import calendar
import argparse
import json

In [2]:
# read the config file
config_file = "/home/muhammed/METEO-REPORTING-SYSTEM/ME_R_SY/config/config_2018_01.yml"
with open(config_file , "r") as f :
    config = yaml.safe_load(f)

In [4]:
year       = config["date"]["year"]
month      = config["date"]["month"]
input_dir  = config["data"]["input"] 
ref_dir    = f"{input_dir}/reference_1991_2020.grib"
output_dir = config["data"]["output"] 
shp_path   = config["data"]["shapefile"]
year       = config["date"]["year"]
month      = config["date"]["month"]

In [ ]:
# read the shapefile
grib_path = f"{input_dir}/input_{year}_{month:02d}.grib"

ref_path  = f"{input_dir}/reference_1991_2020.grib"

cmap_dict = {"2t": "RdYlBu_r", "tp": "GnBu", "msl": "viridis", "w": "YlOrRd"}

cmap_dict_anom = {"2t": "bwr", "tp": "PuOr", "msl": "BrBG", "w": "RdGy"}

shp = gpd.read_file(shp_path)
ds        = get_grib(grib_path)
ds_ref    = get_grib(ref_path)
ref_mean  = list(map(lambda ds : Statistics.ref_avg(ds , month) , ds_ref))
month_range = calendar.monthrange(year , month)[1]

variables = get_var(ds)
ref_variables = get_var(ref_mean)
ref_variables["tp"] = ref_variables["tp"] * month_range

Stats = Statistics()
Plot = Plotter(shp , cmap_dict , cmap_dict_anom)
statistics  = Stats.calculate_statistics(variables = variables)
daily_stats = Stats.calculate_daily(variables)
anomaly     = Stats.calculate_anomaly(statistics , ref_variables)


# III- VIZUALISATION

## III-1 MONTHLY AVERAGE

Plot.plot_monthly_avg(statistics , year , month , output_dir)

## III-2 MONTHLY MAX

Plot.plot_max_days(daily_stats , "max" , year , month , output_dir)

## III-3 MONTHLY MIN

Plot.plot_max_days(daily_stats , "min" , year , month , output_dir)

# III-3 ANOMALY 

Plot.plot_anomaly(anomaly , year , month , output_dir)